# keyboard_config

> Alignment-specific keyboard building blocks for assembly into a shared ZoneManager

In [ ]:
#| default_exp components.step_alignment.keyboard_config

In [ ]:
#| export
from typing import Tuple

# Keyboard navigation library
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.focus_zone import FocusZone

# Card stack library
from cjm_fasthtml_card_stack.core.config import CardStackConfig
from cjm_fasthtml_card_stack.core.html_ids import CardStackHtmlIds
from cjm_fasthtml_card_stack.core.button_ids import CardStackButtonIds
from cjm_fasthtml_card_stack.keyboard.actions import (
    create_card_stack_focus_zone, create_card_stack_nav_actions,
)

## Button IDs

Hidden button IDs triggered by keyboard actions. These buttons are rendered by the keyboard
navigation system and fire HTMX requests to alignment route handlers.

In [ ]:
#| export
SD_ALIGN_TOGGLE_ASSIGN_BTN = "sd-align-toggle-assign-btn"
SD_ALIGN_AUTO_ALIGN_BTN = "sd-align-auto-align-btn"
SD_ALIGN_UNDO_BTN = "sd-align-undo-btn"

## Keyboard Parts Builder

Returns `(zone, actions, modes)` tuple for assembly into a shared `ZoneManager`
by the combined-level keyboard config. No sub-modes needed (unlike decomp's split mode).

In [ ]:
#| export
def create_align_kb_parts(
    ids:CardStackHtmlIds,  # Card stack HTML IDs
    button_ids:CardStackButtonIds,  # Card stack button IDs for navigation
    config:CardStackConfig,  # Card stack configuration
) -> Tuple[FocusZone, tuple, tuple]:  # (zone, actions, modes)
    """Create alignment-specific keyboard building blocks."""
    # Card stack zone from library
    card_zone = create_card_stack_focus_zone(
        ids=ids,
        on_focus_change="onAlignFocusChange",
        data_attributes=("chunk-index", "start-time", "end-time"),
    )
    zone_id = card_zone.id

    # Card stack navigation actions from library (arrows, page jump, first/last, width)
    nav_actions = create_card_stack_nav_actions(
        zone_id=zone_id,
        button_ids=button_ids,
        config=config,
    )

    # Alignment-specific actions (zone-restricted to prevent firing when decomp zone is active)
    align_zone_ids = (zone_id,)

    align_actions = (
        # Toggle assign (Space)
        KeyAction(
            key=" ",
            htmx_trigger=SD_ALIGN_TOGGLE_ASSIGN_BTN,
            zone_ids=align_zone_ids,
            description="Assign/unassign chunk",
            hint_group="Alignment",
        ),

        # Undo (Ctrl+Z, zone-restricted)
        KeyAction(
            key="z",
            modifiers=frozenset({"ctrl"}),
            htmx_trigger=SD_ALIGN_UNDO_BTN,
            zone_ids=align_zone_ids,
            description="Undo",
            hint_group="General",
        ),
    )

    actions = (*nav_actions, *align_actions)
    modes = ()  # No sub-modes — alignment uses a single navigation mode

    return card_zone, actions, modes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()